# 5. Long-Term Memory

Moving from **"remembering this conversation"** to **"remembering across conversations."**

## The problem

Today:
```text
User: My favorite programming language is Java.
```
Tomorrow (new conversation):
```text
User: What programming language do I like?
```
Normal chat history is `history = []` in a new conversation, so the LLM has no idea.

## The fix

Store important info **outside** the chat. Retrieve it in the next conversation.

```text
                 Long-Term Memory
                       ↓
              ┌──────────────────┐
              │ User likes Java   │
              │ User uses Spring  │
              └──────────────────┘
                       ↑
New conversation → retrieve relevant info
                       ↓
                      LLM
```

Storage options: SQL, NoSQL, **vector DB**, files, any persistent storage.

## Key idea

Long-term memory is **NOT** dumping the whole old conversation into the context window.

```text
Old info → Store → New conversation → Retrieve RELEVANT info → Context → LLM
```

Example:
```text
Memory:    "User's preferred language: Java"
Question:  "What should I use for my backend?"

Context given to LLM:
   User prefers Java.
   Question: What should I use for my backend?
```

## Full memory architecture

```text
Current conversation
        ↓
   Chat History
        ↓
 ┌──────────────────┐
 │ What should we   │
 │ send to the LLM? │
 └──────────────────┘
        ↓
 ┌──────────────────┐
 │ Recent messages  │ ← Windowed memory
 │ + summary        │ ← Summarization
 │ + retrieved info │ ← Long-term memory
 └──────────────────┘
        ↓
   Context Window
        ↓
       LLM
```

| Concept | Purpose |
| --- | --- |
| **Chat History** | Store current conversation |
| **Context Window** | Model's max available context |
| **Windowed Memory** | Keep recent messages |
| **Summarization** | Compress older messages |
| **Long-Term Memory** | Persist important info across conversations |

## Proper notes (the 3 steps)

Every long-term memory system does these 3 things:

1. **Extract**: after each user message, decide *"is there a fact worth saving?"* (name, preferences, tech stack, goals). Most messages have nothing to save.
2. **Store**: save the fact (as short text) in persistent storage. If using a vector DB, also save its **embedding**.
3. **Retrieve**: on a new message, find the top-k most *relevant* memories (by similarity search) and put them in the prompt.

**What to store vs not store**

| Store | Don't store |
| --- | --- |
| Preferences ("likes Java") | Small talk ("hi", "thanks") |
| Stable facts ("works on Kafka project") | Full raw conversations |
| Goals ("preparing for SDE-1 interviews") | Passwords / API keys / secrets |

**Things that go wrong (real-world problems)**

- **Duplicates**: user says "I like Java" 5 times → 5 same memories. Fix: check similarity before saving.
- **Outdated facts**: "I like Java" → later "I moved to Go". Fix: update or replace old memory, add timestamps.
- **Wrong retrieval**: irrelevant memory confuses the LLM. Fix: use top-k + minimum similarity score.
- **Privacy**: user must be able to see and delete their memories.

**Short-term vs long-term**

| | Short-term (chat history) | Long-term |
| --- | --- | --- |
| Scope | One conversation | Across conversations |
| Lives in | Python list / RAM | DB / file / vector store |
| Goes to LLM | All (or windowed) | Only relevant parts |

> Long-term memory is basically **RAG over the user's own facts**.

Below is a small working version: JSON file as storage, Gemini embeddings for retrieval.

In [ ]:
import os, json
import numpy as np
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
embedder = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

### Step 2 & 3: Memory store (store + retrieve)

In [ ]:
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


class MemoryStore:
    def __init__(self, path="memory.json"):
        self.path = path
        self.items = json.load(open(path)) if os.path.exists(path) else []

    def _save(self):
        json.dump(self.items, open(self.path, "w"), indent=2)

    def add(self, text, dup_threshold=0.9):
        vec = embedder.embed_query(text)
        # skip if we already have almost the same memory
        for m in self.items:
            if cosine(vec, m["embedding"]) >= dup_threshold:
                return False
        self.items.append({
            "text": text,
            "embedding": vec,
            "created": datetime.now().isoformat(timespec="seconds"),
        })
        self._save()
        return True

    def search(self, query, k=3, min_score=0.5):
        if not self.items:
            return []
        qvec = embedder.embed_query(query)
        scored = [(cosine(qvec, m["embedding"]), m["text"]) for m in self.items]
        scored.sort(reverse=True)
        return [text for score, text in scored[:k] if score >= min_score]

    def clear(self):
        self.items = []
        self._save()

### Step 1: Extract facts worth saving

In [ ]:
def extract_memories(user_msg):
    prompt = f"""From the user message below, extract facts worth remembering long-term
(preferences, tech stack, goals, personal details). Ignore small talk and questions.
Write each fact as a short sentence starting with "User".
Return ONLY a JSON list of strings. If nothing to save, return [].

Message: {user_msg}"""
    out = llm.invoke(prompt).content
    out = out.replace("```json", "").replace("```", "").strip()
    try:
        facts = json.loads(out)
        return facts if isinstance(facts, list) else []
    except json.JSONDecodeError:
        return []

extract_memories("My favorite programming language is Java and I'm building a Kafka project.")

### Chat function: history + retrieved memory

In [ ]:
def chat(user_msg, history, store):
    # 1. retrieve relevant long-term memories
    relevant = store.search(user_msg)

    system = "You are a helpful assistant."
    if relevant:
        system += "\n\nThings you know about the user:\n" + "\n".join(f"- {m}" for m in relevant)

    # 2. send: system (with memories) + current chat history + new message
    messages = [{"role": "system", "content": system}] + history
    messages.append({"role": "user", "content": user_msg})
    res = llm.invoke(messages)

    # 3. update short-term history
    history.append({"role": "user", "content": user_msg})
    history.append({"role": "ai", "content": res.content})

    # 4. update long-term memory
    for fact in extract_memories(user_msg):
        store.add(fact)

    return res.content

## Demo: Conversation 1 (today)

In [ ]:
store = MemoryStore("memory.json")
store.clear()   # start fresh for the demo

history_1 = []
for msg in [
    "Hi! My favorite programming language is Java.",
    "I mostly use Spring Boot for backend work.",
    "Thanks, that's all for now.",
]:
    print("User:", msg)
    print("AI:", chat(msg, history_1, store), "\n")

What got saved in long-term memory:

In [ ]:
for m in store.items:
    print("-", m["text"])

## Demo: Conversation 2 (tomorrow, brand new chat)

`history_2 = []` is empty. The model has **no** chat history, but memory is retrieved from the store.

In [ ]:
history_2 = []   # new conversation

q = "What programming language do I like?"
print("User:", q)
print("AI:", chat(q, history_2, store))

In [ ]:
q = "What should I use for my backend?"
print("User:", q)
print("Retrieved:", store.search(q))
print("AI:", chat(q, history_2, store))

## Quick recap

- Chat history = **this** conversation. Long-term memory = **all** conversations.
- Flow: **Extract → Store → Retrieve → Put in prompt**.
- Only retrieve *relevant* memories, don't dump everything.
- Watch out for duplicates, outdated facts, and privacy.

**Next:** how LangChain / LangGraph implements this (checkpointers for chat history, `Store` for long-term memory).

### Try it yourself
1. Tell the bot "I switched from Java to Go". What happens to the old memory? (Hint: nothing, it's still there. How would you fix it?)
2. Replace the JSON file with a real vector DB (Chroma / FAISS).
3. Add a `delete_memory()` method so the user can remove a memory.